In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 20. Supervised Fine-Tuning (SFT) — Following Instructions

> **Learning Goals**
> - Understand how SFT turns a pretrained model into an instruction-following model
> - Design an instruction-response data format
> - Understand the role of loss masking

## 20.1 Pretraining → SFT → RLHF Pipeline

1. **Pretraining**: next-token prediction on TB-scale data
2. **SFT**: instruction-following training on instruction-response pairs (tens of thousands to millions of samples)
3. **RLHF/DPO**: alignment with human preferences (tens of thousands of samples)

SFT teaches the behavior of answering questions. A pretrained model by itself only continues text.

## 20.2 Instruction-Response Data Format

Example (Alpaca format):
```
Below is an instruction... ### Instruction: {instruction} ### Response: {response}
```

ChatML (modern):
```
<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>
```

## 20.3 Loss Masking — Do Not Train on the Prompt

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|\mathbf{y}|} \sum_{t=1}^{|\mathbf{y}|} \log P(y_t | \mathbf{x}, \mathbf{y}_{<t}; \theta)$$

**Key idea**: exclude prompt tokens $\mathbf{x}$ from the loss. Train only on response tokens $\mathbf{y}$.
- Reason: we do not want the model to learn how to generate the prompt; we want it to generate good responses.


In [ ]:

# Shared model definition, same as Ch 18
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model definition complete")


In [ ]:
# mini SFT dataset
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# simple instruction-response examples; real datasets are larger
sft_examples = [
    {"instruction": "What is 2+2?", "response": "2+2 equals 4."},
    {"instruction": "Translate hello to Korean.", "response": "hello in Korean is annyeonghaseyo."},
    {"instruction": "Capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "Define AI.", "response": "AI is the simulation of human intelligence in machines."},
    {"instruction": "Reverse abc.", "response": "cba"},
]

# simple character-level tokenizer
all_text = " ".join([f"{e['instruction']} {e['response']}" for e in sft_examples])
chars = sorted(set(all_text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
print(f"Vocabulary Size: {vocab_size}")

def encode(text):
    return [char_to_idx[c] for c in text if c in char_to_idx]

def format_example(ex):
    """Alpaca-style format."""
    return f"### Instruction: {ex['instruction']}### Response: {ex['response']}<END>"

# batch with loss masking
def make_sft_batch(examples, max_len=128):
    """instruction 0, response 1 with Mask."""
    inputs, masks = [], []
    for ex in examples:
        formatted = format_example(ex)
        ids = encode(formatted)[:max_len]
        # identify the response portion
        resp_start = formatted.find("### Response:") + len("### Response:")
        full_text = formatted
        # mask: 0 for instruction, 1 for response
        mask = [0] * len(formatted)
        for i in range(resp_start, len(formatted)):
            mask[i] = 1
        # truncate
        mask = mask[:len(ids)]
        # padding
        while len(ids) < max_len:
            ids.append(0)
            mask.append(0)
        inputs.append(ids)
        masks.append(mask)
    return (torch.tensor(inputs, dtype=torch.long),
            torch.tensor(masks, dtype=torch.float32))

x, mask = make_sft_batch(sft_examples, max_len=128)
print(f"Input: {x.shape}, mask: {mask.shape}")
print(f"Mean response-token ratio: {mask.mean():.4f}")


In [ ]:
# SFT training loop
def sft_loss_with_mask(logits, targets, mask):
    """cross-entropy loss with masking applied."""
    # logits: (B, T, V), targets: (B, T), mask: (B, T)
    B, T, V = logits.shape
    # shift: Input [0..T-2], Label [1..T-1]
    logits = logits[:, :-1, :].reshape(-1, V)
    targets = targets[:, 1:].reshape(-1)
    mask = mask[:, 1:].reshape(-1)
    # per-token loss
    losses = F.cross_entropy(logits, targets, reduction='none')
    # Mask Application
    masked_loss = (losses * mask).sum() / (mask.sum() + 1e-8)
    return masked_loss

# Training
model = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=128)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Compare loss masking on vs off
import matplotlib.pyplot as plt

# 200 Step Training (masking on)
torch.manual_seed(0)
model_with_mask = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=128)
opt1 = torch.optim.AdamW(model_with_mask.parameters(), lr=1e-3)
losses_masked = []
for step in range(200):
    x, mask = make_sft_batch(sft_examples, max_len=128)
    opt1.zero_grad()
    logits = model_with_mask(x)
    loss = sft_loss_with_mask(logits, x, mask)
    loss.backward()
    opt1.step()
    losses_masked.append(loss.item())

plt.figure(figsize=(10, 4))
plt.plot(losses_masked, label='SFT with loss masking', color='blue')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('SFT Learning Curve (training only on the response portion)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch20_sft.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Final loss: {losses_masked[-1]:.4f}")


In [ ]:
# SFT after response generation
def generate_sft(model, instruction, max_new=80, temperature=0.7):
    model.eval()
    prompt = f"### Instruction: {instruction}### Response:"
    idx = encode(prompt)
    with torch.no_grad():
        for _ in range(max_new):
            x = torch.tensor([idx[-128:]], dtype=torch.long)
            logits = model(x)
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            if next_idx == char_to_idx.get('<', -1):
                break
            idx.append(next_idx)
            # stop at <END> (simple version)
            text_so_far = ''.join(idx_to_char.get(i, '') for i in idx)
            if '<END>' in text_so_far:
                break
    return ''.join(idx_to_char.get(i, '') for i in idx)

print("=== Responses after SFT ===\n")
for inst in ["What is 2+2?", "Capital of France?", "Reverse abc."]:
    print(f"Q: {inst}")
    print(f"A: {generate_sft(model_with_mask, inst, max_new=80, temperature=0.5)}")
    print()


## 20.4 Data Quality Determines Performance — Lesson from LIMA

LIMA (Meta, 2023) experiment:
- Pretrained model + only 1,000 high-quality instruction-response examples for SFT
- A 65B LLaMA model produced responses comparable to GPT-4-level systems in many settings

Lesson: **quality matters more than quantity**. Pretraining already teaches much of the knowledge. SFT often only needs to teach the response format.

## 20.5 Full Fine-Tuning vs PEFT (Parameter-Efficient Fine-Tuning)

- **Full fine-tuning**: updates all parameters. Expensive.
- **PEFT**: updates only a small subset of parameters (LoRA, Adapter, Prefix Tuning, etc.). Covered in Ch 26.

## 20.6 Key Takeaways

| Concept | Description |
|---|---|
| SFT | instruction-following training with instruction-response pairs |
| Loss masking | exclude prompt tokens from the loss |
| Learning rate | roughly 1/10 of pretraining |
| Data quality | quality over quantity (LIMA lesson) |

## Exercises

1. Train with loss masking on and off, then compare response quality.
2. Run SFT with learning rates 1e-2, 1e-3, and 1e-4.
3. Increase instruction data from 5 to 50 to 500 examples and compare the SFT effect.
4. Convert SFT data to ChatML format.
5. Compare responses from a pretrained model and an SFT model, then explain the difference.

> Solutions: `solutions/ch20_solutions.ipynb`
